# Soft-barrier primitive skeleton experiment

This notebook mirrors the main skeleton tutorial, but inserts an experimental body/branch ownership step based on soft-barrier primitive fitting. The body uses a mildly flattenable superellipsoid, while branches remain ellipsoids. The goal is to identify body- and branch-owned vertices before those vertices are allowed to become crypt components.

The default skeleton pipeline is unchanged, but this notebook enables an opt-in body barrier hook inside `detect_crypts_for_skeleton` so the body-owned vertices are protected before crypt refinement. Branch protection is still tested as a notebook-level adapter after split candidates exist.


In [1]:
# --- Imports ---
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import HTML, display

from organograph.io_utils.dataset_config import load_mesh_dataset_config
from organograph.io_utils.path_parsing import discover_mesh_paths, parse_mesh_path
from organograph.mesh.OrganoidMesh import OrganoidMesh
from organograph.mesh.geodesics import compute_geodesics_dijkstra
from organograph.crypts.filters import filter_crypts_by_hks_percent, filter_crypts_by_size

# During development, Jupyter may keep older skeleton and plotting modules
# in memory. Drop both so the renderer recognizes newly added primitive types.
import importlib
import sys
for _mod in list(sys.modules):
    if _mod.startswith("organograph.skeleton") or _mod == "organograph.plotting.skeletons":
        sys.modules.pop(_mod, None)
importlib.invalidate_caches()

from organograph.skeleton import (
    PrimitiveAttachment,
    PrimitiveFitConfig,
    SkeletonizationConfig,
    SkeletonizationResult,
    build_skeleton_from_crypt_detections,
    detect_crypts_for_skeleton,
    barrier_primitive_vertices_like_mesh,
    fit_primitives_for_skeletonization_result,
    protect_detection_regions_from_mask,
)
from organograph.plotting.skeletons import plot_mesh_with_skeleton_and_primitives


## 1) Dataset paths and organoid selection

Use either the manual `ORGANOID_SPECS` list or turn on automatic discovery.


In [ ]:
# -----------------------
# CONFIG: edit these
# -----------------------
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "notebooks" and (NOTEBOOK_DIR / "notebooks").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
PROJECT_ROOT = NOTEBOOK_DIR.parent

DATASET = "20251201"
MESH_DATA_DIR = PROJECT_ROOT.parent / "NicoleData" / DATASET / "fractal_output"
MESH_CONFIG_PATH = PROJECT_ROOT.parent / "NicoleData" / DATASET / "mesh_config.json"
VOCAB_PATH = PROJECT_ROOT / "sim" / "vocab_with_meta.npz"

USE_AUTO_DISCOVERY = True
AUTO_TIMEPOINT = "day4p5"
AUTO_N_ORGANOIDS = 5
AUTO_WELLS = None  # e.g. ["B02", "B03"]
AUTO_RANDOM_SEED = 69

ORGANOID_SPECS = [
    # {"timepoint": "day4p5", "well": "B03", "organoid_id": "144"},
    # {"timepoint": "day4p5", "well": "B02", "organoid_id": "124"},
    # {"timepoint": "day4p5", "well": "B04", "organoid_id": "4"},
    # {"timepoint": "day4p5", "well": "B05", "organoid_id": "54"},
    # {"timepoint": "day4p5", "well": "B02", "organoid_id": "115"},
    # {"timepoint": "day4p5", "well": "B02", "organoid_id": "100"},
    # {"timepoint": "day4p5", "well": "B02", "organoid_id": "31"},
    # {"timepoint": "day4p5", "well": "B02", "organoid_id": "39"},
]
MESH_PATH_OVERRIDES = {}

mesh_cfg = load_mesh_dataset_config(str(MESH_CONFIG_PATH))

if USE_AUTO_DISCOVERY:
    wells = {AUTO_TIMEPOINT: list(AUTO_WELLS)} if AUTO_WELLS else None
    discovered = discover_mesh_paths(
        data_dir=str(MESH_DATA_DIR),
        timepoints=[AUTO_TIMEPOINT],
        zarr_names=mesh_cfg["zarr_name_by_tp"],
        rounds=mesh_cfg["round_by_tp"],
        meshes=mesh_cfg["meshname_by_tp"],
        wells=wells,
    )
    if AUTO_RANDOM_SEED is not None:
        rng = np.random.default_rng(int(AUTO_RANDOM_SEED))
        discovered = list(rng.permutation(discovered))
    selected = list(discovered[: int(AUTO_N_ORGANOIDS)])
    ORGANOID_SPECS = []
    MESH_PATH_OVERRIDES = {}
    for mesh_path in selected:
        parsed = parse_mesh_path(mesh_path)
        spec = {
            "timepoint": parsed["timepoint"],
            "well": parsed["well"],
            "organoid_id": parsed["organoid_id"],
        }
        ORGANOID_SPECS.append(spec)
        MESH_PATH_OVERRIDES[f"{spec['well']}/{spec['organoid_id']}"] = mesh_path

print("PROJECT_ROOT     =", PROJECT_ROOT)
print("MESH_DATA_DIR   =", MESH_DATA_DIR)
print("MESH_CONFIG_PATH=", MESH_CONFIG_PATH)
print("VOCAB_PATH      =", VOCAB_PATH)
print("n_organoids     =", len(ORGANOID_SPECS))


PROJECT_ROOT     = /home/fmoller/Projects/LearningOrganoids/OrganoGraph
MESH_DATA_DIR   = /home/fmoller/Projects/LearningOrganoids/NicoleData/20251201/fractal_output
MESH_CONFIG_PATH= /home/fmoller/Projects/LearningOrganoids/NicoleData/20251201/mesh_config.json
VOCAB_PATH      = /home/fmoller/Projects/LearningOrganoids/OrganoGraph/sim/vocab_with_meta.npz
n_organoids     = 5


In [3]:
def mesh_path_from_spec(spec):
    timepoint = str(spec["timepoint"])
    well = str(spec["well"])
    organoid_id = str(spec["organoid_id"])
    override_key = f"{well}/{organoid_id}"
    if override_key in MESH_PATH_OVERRIDES:
        return Path(MESH_PATH_OVERRIDES[override_key])
    return (
        Path(MESH_DATA_DIR)
        / timepoint
        / mesh_cfg["zarr_name_by_tp"][timepoint]
        / well[0]
        / well[1:]
        / mesh_cfg["round_by_tp"][timepoint]
        / "meshes"
        / mesh_cfg["meshname_by_tp"][timepoint]
        / f"{organoid_id}.vtp"
    )

organoid_records = []
missing = []
for spec in ORGANOID_SPECS:
    mesh_path = mesh_path_from_spec(spec)
    rec = dict(spec)
    rec["mesh_path"] = str(mesh_path)
    rec["label_uid"] = f"{rec['timepoint']}_{rec['well']}_{rec['organoid_id']}"
    if mesh_path.exists():
        try:
            parsed = parse_mesh_path(str(mesh_path))
            rec["label_uid"] = parsed.get("label_uid", rec["label_uid"])
        except Exception:
            pass
        organoid_records.append(rec)
    else:
        missing.append(rec)

if missing:
    print("Missing requested mesh paths:")
    for rec in missing:
        print(" ", rec["mesh_path"])

print("usable records =", len(organoid_records))
for rec in organoid_records:
    print(" ", rec["label_uid"], rec["mesh_path"])


usable records = 5
  day4p5_B02_100 /home/fmoller/Projects/LearningOrganoids/NicoleData/20251201/fractal_output/day4p5/251130R0.zarr/B/02/2_zillum_registered/meshes/nnorg_corrected_smoothed_annotated_by_projection/100.vtp
  day4p5_B02_101 /home/fmoller/Projects/LearningOrganoids/NicoleData/20251201/fractal_output/day4p5/251130R0.zarr/B/02/2_zillum_registered/meshes/nnorg_corrected_smoothed_annotated_by_projection/101.vtp
  day4p5_B02_11 /home/fmoller/Projects/LearningOrganoids/NicoleData/20251201/fractal_output/day4p5/251130R0.zarr/B/02/2_zillum_registered/meshes/nnorg_corrected_smoothed_annotated_by_projection/11.vtp
  day4p5_B02_110 /home/fmoller/Projects/LearningOrganoids/NicoleData/20251201/fractal_output/day4p5/251130R0.zarr/B/02/2_zillum_registered/meshes/nnorg_corrected_smoothed_annotated_by_projection/110.vtp
  day4p5_B02_112 /home/fmoller/Projects/LearningOrganoids/NicoleData/20251201/fractal_output/day4p5/251130R0.zarr/B/02/2_zillum_registered/meshes/nnorg_corrected_smoothed_

## 2) Mesh preparation

If smoothing is enabled, the smoothed mesh is used for detection, barrier fitting, skeleton construction, primitive fitting, and plotting.


In [4]:
NORMALIZE_MESH = True
NORMALIZE_SCALE = 10.0
EIGEN_K = 225

SMOOTH_MESH = True
SMOOTH_LMAX = 12
SMOOTH_EIGEN_K = None

vocab = np.load(str(VOCAB_PATH), allow_pickle=True)
MESH_CACHE = {}


def _clamped_eigen_k(mesh, requested_k):
    n_vertices = int(np.asarray(mesh.v).shape[0])
    return max(2, min(int(requested_k), n_vertices - 2))


def _reset_spectral_state(mesh):
    mesh.laplacian = None
    mesh.mass_matrix = None
    mesh.eigvals = None
    mesh.eigvecs = None
    mesh.coeffs_v = None
    mesh.lmax = None


def _ensure_mesh_eigendecomposition(mesh, requested_k):
    k = _clamped_eigen_k(mesh, requested_k)
    if mesh.eigvals is None or mesh.eigvecs is None or mesh.eigvecs.shape[1] < k:
        _reset_spectral_state(mesh)
        mesh._eig_decomp(k=k)
    return k


def _smooth_mesh_low_pass(mesh):
    lmax = int(SMOOTH_LMAX)
    coeff_k = int(lmax ** 2)
    _ensure_mesh_eigendecomposition(mesh, max(EIGEN_K, coeff_k))
    mesh.compute_spectral_coefficients(lmax=lmax)
    mesh.v = np.asarray(mesh.reconstruct_from_coeffs(mesh.coeffs_v, lmax=lmax), dtype=float)
    _reset_spectral_state(mesh)
    _ensure_mesh_eigendecomposition(mesh, SMOOTH_EIGEN_K or EIGEN_K)
    return mesh


def load_prepared_mesh(record):
    label_uid = record["label_uid"]
    if label_uid in MESH_CACHE:
        return MESH_CACHE[label_uid]
    mesh = OrganoidMesh(str(record["mesh_path"]))
    mesh.label_uid = label_uid
    if NORMALIZE_MESH:
        mesh.normalize_inplace(scale=NORMALIZE_SCALE, center="mean")
    if SMOOTH_MESH:
        _smooth_mesh_low_pass(mesh)
    else:
        _ensure_mesh_eigendecomposition(mesh, EIGEN_K)
    MESH_CACHE[label_uid] = mesh
    return mesh


## 3) Tuning parameters

`BARRIER_KWARGS` controls the body superellipsoid, which is fitted before HKS candidate detection. `epsilon_1 < 1` gives the body flatter ends and fuller sides. `BODY_RELATIVE_HEIGHT_THRESHOLD` marks vertices as body-owned from its radial level. After split hierarchy is known, branch barriers remain ellipsoids and are fitted from body-trimmed split regions. Attachments are assigned from the first persistent crossing of geodesic ring centers with the corresponding host primitive; circumference profiles still classify genuine constrictions.


In [5]:
# --- Crypt detection parameters ---
DETECTION_KWARGS = dict(
    L_ref=None,
    crypt_vocab_idx=None,
    threshold=0.5,
    refine_crypts=True,
    refine_threshold=0.00,
    refine_only_if_area_at_least=5.0,
    min_refined_frac_of_parent=0.05,
    geodesic_kwargs=None,
    final_tip_hks_time=1.0,
    final_tip_bottom_fraction=0.6,
    final_tip_min_hks_percent_increase=5.0,
    extend_max=2.0,
    disc_resolution=200,
    neck_search_interval=(0.8, 2.0),
    neck_window_length=9,
    neck_polyorder=3,
    neck_min_prominence=0.05,
    neck_min_length=0.05,
    validate_split_stems=True,
    validate_branch_geometry=True,
    branch_min_confidence=0.85,
    branch_max_neck_to_body_radius_ratio=0.70,
    split_growth_max_size_factor=3.0,
    split_growth_max_mesh_fraction=0.40,
    split_growth_smooth_perimeter=True,
    split_growth_smoothing_tolerance=0.0,
    split_growth_min_decrease_fraction=0.0,
    split_growth_min_prominence_fraction=0.01,
    split_growth_robust_window=1,
    refine_broad_crypt_openings=False,
    refine_body_transition_width_outliers=True,
    body_transition_max_crypt_to_host_width_ratio=0.80,
    body_transition_host_width_quantile=0.75,
    body_transition_min_second_derivative_score=0.60,
    body_transition_min_attachment_level=0.25,
)

FILTER_KWARGS = dict(
    use_hks_filter=True,
    min_percent_greater=1.0,
    hks_t_min=None,
    hks_t_max=None,
    use_size_filter=True,
    min_patch_area=5.0,
)

BUILD_KWARGS = dict(
    body_center=None,
    bend_strategy="crypt_centroid",
    bend_max_dimensionless_curvature=0.50,
    bend_curvature_penalty=8.0,
    refine_body_center_from_necks=True,
    refine_branch_centers_from_necks=True,
)

BARRIER_KWARGS = dict(
    primitive_type="superellipsoid",
    barrier_weight=2.5, # 150
    underfill_weight=0.02, # 0.50
    center_regularization=0.01,
    anisotropy_regularization=0.1,
    center_shift_limit_frac=0.55,
    initial_radius_quantile=0.6,
    initial_epsilon_1=0.9,
    epsilon_1_bounds=(0.35, 1.0),
    epsilon_1_regularization=0.01,
    epsilon_2=1.0,
    maxiter=2000,
)
BRANCH_BARRIER_KWARGS = dict(BARRIER_KWARGS)
BRANCH_BARRIER_KWARGS["primitive_type"] = "ellipsoid"
BRANCH_BARRIER_KWARGS["anisotropy_regularization"] = 1.0
BRANCH_BARRIER_KWARGS["underfill_weight"] = 0.5
BRANCH_BARRIER_KWARGS["barrier_weight"] = 100


BODY_RELATIVE_HEIGHT_THRESHOLD = 1.2
BRANCH_RELATIVE_HEIGHT_THRESHOLD = 1.1
BARRIER_SAMPLE_FRACTION = 1.0
BARRIER_SAMPLE_SEED = 0
USE_BODY_BARRIER_DURING_DETECTION = True
BODY_BARRIER_MIN_CANDIDATE_VERTICES = 4
MIN_BRANCH_BARRIER_VERTICES = 20

USE_INITIAL_BARRIER_BLOBS_FOR_BODY_BRANCH = True
ASSIGN_ATTACHMENTS_FROM_BARRIER_CROSSINGS = True
BARRIER_CROSSING_KWARGS = dict(
    surface_level=1.0,
    min_axis_level=0.03,
    max_axis_level=2.0,
    n_samples=40,
    persistence=2,
    bisection_iterations=8,
)

PRIMITIVE_CONFIG = PrimitiveFitConfig(
    body_branch_neck_kwargs=dict(
        radius_quantile=0.5,
        expansion_factor=1.35,
        max_extent_fraction=0.25,
        min_extent_radius_fraction=0.35,
    ),
    body_kwargs=dict(
        primitive_type="asymmetric_superellipsoid",
        add_attachment_cap_support=True,
        cap_support_points_per_attachment=64,
        cap_support_radius_fraction=0.5,
    ),
    branch_kwargs=dict(primitive_type="asymmetric_superellipsoid"),
    crypt_tube_kwargs=dict(
        smooth_centerline=True,
        smooth_bulged_centerlines=False,
        centerline_n_bands=7,
        centerline_n_samples=64,
        centerline_constriction_weight=4.0,
        update_crypt_nodes=True,
        radius_quantile=0.5,
        optimize_radius_profile=True,
        initial_body_position=0.5,
        initial_taper_position=0.85,
        body_position_bounds=(0.2, 0.7),
        min_taper_gap=0.1,
        max_taper_position=0.9,
    ),
)


def make_filter_list(**kw):
    filters = []
    if kw.get("use_hks_filter", False):
        filters.append(
            lambda patches, **inner: filter_crypts_by_hks_percent(
                patches,
                min_percent_greater=kw.get("min_percent_greater", 2.0),
                t_min=kw.get("hks_t_min"),
                t_max=kw.get("hks_t_max"),
                **inner,
            )
        )
    if kw.get("use_size_filter", False):
        filters.append(
            lambda patches, **inner: filter_crypts_by_size(
                patches,
                min_patch_verts=kw.get("min_patch_verts", 0),
                min_patch_area=kw.get("min_patch_area", 5.0),
                **inner,
            )
        )
    return filters or None


## 4) Barrier primitive helpers

The body barrier is fit to the whole organoid mesh. Branch barriers are fit to rough split parent regions after removing body-owned vertices; those branch masks are only used to protect daughter crypt components in this experimental notebook.


In [6]:
def _protected_mask_values(mesh, body_mask, branch_masks=None):
    values = np.zeros(mesh.v.shape[0], dtype=int)
    if body_mask is not None:
        values[np.asarray(body_mask, dtype=bool)] = 1
    for branch_mask in (branch_masks or {}).values():
        values[np.asarray(branch_mask, dtype=bool)] = 2
    return values


def plot_barrier_ellipsoid(
    mesh,
    fit,
    branch_fits=None,
    *,
    body_mask=None,
    branch_masks=None,
    title="Soft-barrier primitives",
):
    body_primitive_vertices = barrier_primitive_vertices_like_mesh(mesh.v, fit)
    mask_values = _protected_mask_values(mesh, body_mask, branch_masks)
    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}]],
        subplot_titles=("mesh + barrier primitives", "protected body/branch mask"),
    )
    fig.add_trace(
        go.Mesh3d(
            x=mesh.v[:, 0], y=mesh.v[:, 1], z=mesh.v[:, 2],
            i=mesh.f[:, 0], j=mesh.f[:, 1], k=mesh.f[:, 2],
            color="lightgray", opacity=0.25, flatshading=True,
            name="mesh", showscale=False, showlegend=False,
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Mesh3d(
            x=body_primitive_vertices[:, 0], y=body_primitive_vertices[:, 1], z=body_primitive_vertices[:, 2],
            i=mesh.f[:, 0], j=mesh.f[:, 1], k=mesh.f[:, 2],
            color="#4c78a8", opacity=0.52, flatshading=True,
            name=f"body {fit.primitive_type}", showscale=False, showlegend=False,
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter3d(
            x=[fit.center[0]], y=[fit.center[1]], z=[fit.center[2]],
            mode="markers", marker=dict(size=5, color="black"),
            name="body center", showlegend=False,
        ),
        row=1, col=1,
    )
    for branch_id, branch_fit in (branch_fits or {}).items():
        branch_vertices = barrier_primitive_vertices_like_mesh(mesh.v, branch_fit)
        fig.add_trace(
            go.Mesh3d(
                x=branch_vertices[:, 0], y=branch_vertices[:, 1], z=branch_vertices[:, 2],
                i=mesh.f[:, 0], j=mesh.f[:, 1], k=mesh.f[:, 2],
                color="#f58518", opacity=0.48, flatshading=True,
                name=str(branch_id), showscale=False, showlegend=False,
            ),
            row=1, col=1,
        )
        fig.add_trace(
            go.Scatter3d(
                x=[branch_fit.center[0]], y=[branch_fit.center[1]], z=[branch_fit.center[2]],
                mode="markers", marker=dict(size=5, color="#8c510a"),
                name=f"{branch_id} center", showlegend=False,
            ),
            row=1, col=1,
        )
    fig.add_trace(
        go.Mesh3d(
            x=mesh.v[:, 0], y=mesh.v[:, 1], z=mesh.v[:, 2],
            i=mesh.f[:, 0], j=mesh.f[:, 1], k=mesh.f[:, 2],
            intensity=mask_values,
            colorscale=[
                [0.0, "lightgray"], [0.32, "lightgray"],
                [0.34, "#4c78a8"], [0.66, "#4c78a8"],
                [0.68, "#f58518"], [1.0, "#f58518"],
            ],
            cmin=0, cmax=2, showscale=False, opacity=1.0, flatshading=True,
            name="protected mask", showlegend=False,
        ),
        row=1, col=2,
    )
    fig.update_layout(
        title=title,
        height=560,
        width=1120,
        margin=dict(l=0, r=20, b=0, t=45),
        scene=dict(aspectmode="data"),
        scene2=dict(aspectmode="data"),
    )
    return fig


def barrier_results_from_intermediates(intermediates):
    info = (intermediates or {}).get("body_barrier_ellipsoid") or {}
    if info.get("fit") is None or info.get("mask") is None:
        raise RuntimeError("Barrier-aware detection did not return a body fit and mask")
    return {
        "body_fit": info["fit"],
        "body_mask": info["mask"],
        "branch_fits": dict(info.get("branch_fits", {})),
        "branch_masks": dict(info.get("branch_masks", {})),
        "protected_mask": info.get("protected_mask", info["mask"]),
        "branch_fit_info": list(info.get("branch_fit_info", [])),
        "crossing_config": dict(info.get("crossing_config", {})),
    }


def barrier_fit_to_attachment(fit, *, attachment_id, target_id, component):
    return PrimitiveAttachment(
        primitive_type=fit.primitive_type,
        parameters=fit.to_primitive_parameters(),
        fit_error=fit.objective,
        residuals={"objective": fit.objective},
        metadata={
            "component": component,
            "source": f"initial_soft_barrier_{fit.primitive_type}",
            "fit_success": bool(fit.success),
            "fit_message": fit.message,
        },
        attachment_type="node",
        attachment_id=attachment_id,
        target_ids=[target_id],
    )


def use_initial_barrier_blob_primitives(primitive_result, barrier):
    graph = primitive_result.graph
    body_attachment = barrier_fit_to_attachment(
        barrier["body_fit"],
        attachment_id="body",
        target_id=graph.body_node().node_id,
        component="body",
    )
    graph.body_node().primitive_attachment = body_attachment
    primitive_result.attachments["body"] = body_attachment

    branch_attachments = {}
    for branch_node_id, branch_fit in barrier.get("branch_fits", {}).items():
        if branch_node_id not in graph.nodes:
            continue
        branch_attachment = barrier_fit_to_attachment(
            branch_fit,
            attachment_id=branch_node_id,
            target_id=branch_node_id,
            component="branch",
        )
        graph.node(branch_node_id).primitive_attachment = branch_attachment
        branch_attachments[branch_node_id] = branch_attachment
    primitive_result.attachments["branches"] = branch_attachments
    return primitive_result


## 5) Run the experimental pipeline

For each organoid:

1. run current crypt detection with optional body barrier protection before refinement;
2. reuse or fit the body barrier superellipsoid;
3. fit branch barrier ellipsoids from rough split-parent regions;
4. remove protected body/branch vertices from crypt-side region fields;
5. build the skeleton and fit final primitives.


In [7]:
def run_barrier_pipeline(record, *, show_plots=True):
    mesh = load_prepared_mesh(record)
    detection_kwargs = dict(DETECTION_KWARGS)
    detection_kwargs["filter_fn_list"] = make_filter_list(**FILTER_KWARGS)
    detection_kwargs.update(
        body_barrier_ellipsoid=USE_BODY_BARRIER_DURING_DETECTION,
        body_barrier_config=BARRIER_KWARGS,
        body_barrier_relative_height_threshold=BODY_RELATIVE_HEIGHT_THRESHOLD,
        body_barrier_min_candidate_vertices=BODY_BARRIER_MIN_CANDIDATE_VERTICES,
        body_barrier_sample_fraction=BARRIER_SAMPLE_FRACTION,
        body_barrier_sample_seed=BARRIER_SAMPLE_SEED,
        barrier_boundary_attachments=ASSIGN_ATTACHMENTS_FROM_BARRIER_CROSSINGS,
        barrier_crossing_kwargs=BARRIER_CROSSING_KWARGS,
        branch_barrier_config=BRANCH_BARRIER_KWARGS,
        branch_barrier_relative_height_threshold=BRANCH_RELATIVE_HEIGHT_THRESHOLD,
        branch_barrier_min_candidate_vertices=MIN_BRANCH_BARRIER_VERTICES,
    )
    detections, intermediates = detect_crypts_for_skeleton(
        mesh,
        vocab,
        geodesic_fn=compute_geodesics_dijkstra,
        return_intermediates=True,
        **detection_kwargs,
    )

    barrier = barrier_results_from_intermediates(intermediates)
    barrier_adjusted_detections = detections

    protected_detections = protect_detection_regions_from_mask(
        barrier_adjusted_detections,
        barrier["protected_mask"],
        metadata_key="barrier_ellipsoid_protection",
    )
    graph_build_kwargs = dict(BUILD_KWARGS)
    if USE_INITIAL_BARRIER_BLOBS_FOR_BODY_BRANCH:
        graph_build_kwargs["body_center"] = barrier["body_fit"].center.tolist()
        graph_build_kwargs["branch_center_overrides"] = {
            branch_node_id: branch_fit.center.tolist()
            for branch_node_id, branch_fit in barrier["branch_fits"].items()
        }

    graph = build_skeleton_from_crypt_detections(
        vertices=mesh.v,
        faces=mesh.f,
        crypt_detections=protected_detections,
        **graph_build_kwargs,
    )
    skeleton_result = SkeletonizationResult(
        graph=graph,
        detections=protected_detections,
        intermediates={
            **intermediates,
            "barrier_ellipsoid": barrier,
            "barrier_adjusted_detections": barrier_adjusted_detections,
        },
        config=SkeletonizationConfig(detection_kwargs=detection_kwargs, build_kwargs=graph_build_kwargs),
        metadata={"record": dict(record), "label_uid": record["label_uid"]},
        mesh=mesh,
    )
    primitive_result = fit_primitives_for_skeletonization_result(
        skeleton_result,
        config=PRIMITIVE_CONFIG,
    )
    if USE_INITIAL_BARRIER_BLOBS_FOR_BODY_BRANCH:
        primitive_result = use_initial_barrier_blob_primitives(primitive_result, barrier)

    if show_plots:
        display(
            plot_barrier_ellipsoid(
                mesh,
                barrier["body_fit"],
                barrier["branch_fits"],
                body_mask=barrier["body_mask"],
                branch_masks=barrier["branch_masks"],
                title=f"{record['label_uid']} body/branch barrier primitives and masks",
            )
        )
        display(
            plot_mesh_with_skeleton_and_primitives(
                mesh.v,
                mesh.f,
                primitive_result.graph,
                backend="plotly",
                mesh_alpha=0.12,
                primitive_alpha=0.32,
                show_node_labels=False,
            )
        )

    return {
        "record": record,
        "mesh": mesh,
        "detections": detections,
        "barrier_adjusted_detections": barrier_adjusted_detections,
        "protected_detections": protected_detections,
        "barrier": barrier,
        "skeleton": skeleton_result,
        "primitives": primitive_result,
    }


barrier_results = []
for record in organoid_records:
    display(HTML(f"<h3>{record['label_uid']}</h3>"))
    result = run_barrier_pipeline(record, show_plots=True)
    barrier_results.append(result)
    barrier = result["barrier"]
    print("body primitive:", barrier["body_fit"].primitive_type)
    print("body primitive success:", barrier["body_fit"].success)
    print("body radii:", np.round(barrier["body_fit"].radii, 4))
    print("body epsilon_1:", round(float(barrier["body_fit"].epsilon_1), 4))
    print("body-owned vertices:", int(np.count_nonzero(barrier["body_mask"])))
    print("protected vertices:", int(np.count_nonzero(barrier["protected_mask"])))
    print("branch barriers:", list(barrier["branch_fits"]))


body primitive: superellipsoid
body primitive success: True
body radii: [4.1906 3.5506 3.4388]
body epsilon_1: 0.9962
body-owned vertices: 14570
protected vertices: 14570
branch barriers: []


body primitive: superellipsoid
body primitive success: True
body radii: [3.3332 3.2612 2.863 ]
body epsilon_1: 0.8312
body-owned vertices: 18001
protected vertices: 18001
branch barriers: []


KeyboardInterrupt: 

## Notes for deployment

This notebook uses the new source helpers in `organograph.skeleton.barrier_ellipsoid`. Body ownership can now be applied inside `detect_crypts_for_skeleton` immediately after initial parent candidate detection and before local refinement/neck computations. Branch ownership is still an experimental notebook-level adapter because branch candidates only exist after split refinement.
